# Chapter 3 - Lab 3: <font color='blue'>CrewAI Multi-Agent Crew</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using CrewAI** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

Where LangChain and ADK gave you a single agent with tools, CrewAI takes a different stance: **model the problem as a crew of specialised agents collaborating on a sequence of tasks**. For our P/E comparison you will define four roles — a Manager, a Data Analyst, a Quant, and an Analyst — and three sequential tasks that pass results between them.

This is the *first multi-agent* lab of the chapter, and it foreshadows the architectural styles you will study in Chapter 7.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **CrewAI** (`crewai`) — role/task/crew abstraction for collaborative agents.
* **OpenAI** `gpt-4o-mini` (default) — drives each agent's reasoning.
* **Sequential task graph** — tasks pass `context` to one another.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q crewai pydantic python-dotenv

## 2. Set up the API key(s)

If you are running this notebook in **Google Colab**, add your OpenAI key in the left vertical menu (the *key* icon) under the name `OPENAI_API_KEY`.

If you are running locally, replace the cell below with `os.environ['OPENAI_API_KEY'] = '...'` or load it from a `.env` file.

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Define the agents (roles)

Each agent in a CrewAI crew is described by a **role**, a **goal**, and a **backstory**. The backstory shapes the agent's persona — what kind of analyst it is, what it cares about. Think of these as concise hiring briefs.

In [ ]:
from crewai import Agent, Crew, Task

manager = Agent(
    role='Manager',
    goal='Coordinate analysis',
    backstory='You are a portfolio manager.',
)

data_agent = Agent(
    role='Data Analyst',
    goal='Fetch stock prices and EPS',
    backstory='You retrieve and clean financial data.',
)

compute_agent = Agent(
    role='Quant',
    goal='Compute ratios',
    backstory='You calculate P/E ratios and prepare metrics.',
)

narrative_agent = Agent(
    role='Analyst',
    goal='Write memo',
    backstory='You explain investment insights in plain English.',
)

## 5. Define the tasks

Tasks form a small **DAG** — `task2` lists `task1` in its `context`, so it receives `task1`'s output. `task3` chains off `task2`. This declarative wiring is CrewAI's way of expressing 'who does what, in what order, with what inputs'.

In [ ]:
task1 = Task(
    description='Get AAPL and JPM prices and EPS',
    agent=data_agent,
    expected_output='Prices and EPS for AAPL and JPM',
)
task2 = Task(
    description='Compute P/E ratios',
    agent=compute_agent,
    context=[task1],
    expected_output='P/E ratios for AAPL and JPM',
)
task3 = Task(
    description='Summarize findings in a memo',
    agent=narrative_agent,
    context=[task2],
    expected_output='Investment memo comparing AAPL and JPM',
)

## 6. Assemble the crew and run it

Pass the agents and the tasks into a `Crew`, then `kickoff()`. CrewAI handles the execution order, the context propagation, and the LLM calls behind the scenes.

In [ ]:
crew = Crew(
    agents=[manager, data_agent, compute_agent, narrative_agent],
    tasks=[task1, task2, task3],
)

result = crew.kickoff()
print(result)

## 7. Results

You should see the agent call `get_stock_data` twice (once per ticker), then call `compute_pe` twice to compute each P/E ratio, and finally produce a short memo comparing the two.

**What to notice about CrewAI specifically:**

* You did not have to write the orchestration loop yourself — the *crew* abstraction encodes the workflow.
* Specialised roles give each LLM call a focused prompt, which can yield higher-quality output than a single generalist agent.
* Trade-off: more LLM calls (≥3 here) means more tokens, latency, and cost. The next chapter on multi-agent systems goes deeper into when this trade is worth it.
* You will see CrewAI's *patterns* (specialist roles, sequential pipelines) again in Chapters 7 and 9, framed as architectural styles rather than a specific framework feature.